# DS Workbench — skills walkthrough

A hands-on tour of the skills the way a data scientist uses them: in a notebook, on DataFrames,
in the order the work actually happens. Everything runs **locally on synthetic data — no
Databricks needed**.

| Part | What you do | Needs Claude? |
|---|---|---|
| 1. The deterministic checks | leakage scan, join check, A/B sample-ratio + CUPED | no — free, offline |
| 2. A predictive workflow | "Who will churn?" — data quality → EDA → leakage → features → model choice → go/no-go | yes |
| 3. A causal investigation | "Did the spring campaign work, and should we expand it?" — frame → find data → verify → plan → analyse → decide | yes |
| 4. The rest of the toolbox | join auditor, A/B traps, difference-in-differences, NL → PySpark, MLOps review, experiment readouts | yes |
| 5. Moving to real data | what changes on Databricks | — |

Cells marked **💲** call Claude (a few cents each; the whole notebook is roughly $1–2). Without
Claude configured, part 1 still works and the 💲 cells show a skill error card instead of results.

**Setup (once):**
```bash
python3 -m venv .venv
.venv/bin/pip install -r requirements.txt jupyter      # or open this file in VS Code
cp .env.example .env                                    # set ANTHROPIC_API_KEY, or an approved route (below)
.venv/bin/jupyter lab notebooks/skills_walkthrough.ipynb
```
Leave `LOCAL_DEV_MODE=true` and `CATALOG_BACKEND=synthetic` (the defaults). If your organisation
reaches Claude through a cloud platform rather than an Anthropic key, set `ANTHROPIC_PROVIDER`
(`bedrock`, `vertex`, `foundry`, …) and, if needed, `ANTHROPIC_BASE_URL` in `.env`.

In [ ]:
import os, sys
from pathlib import Path

# Find the repo root (the folder with config.py) and make it importable
REPO = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "config.py").exists())
sys.path.insert(0, str(REPO))
os.chdir(REPO)                      # so config.py finds .env

import numpy as np
import pandas as pd
from IPython.display import display

from config import config
HAVE_CLAUDE = bool(config.anthropic_api_key) or config.anthropic_provider != "anthropic"

print(f"repo:            {REPO}")
print(f"local dev mode:  {config.local_dev_mode}  (synthetic data, no Databricks)")
print(f"catalog backend: {config.catalog_backend}")
print(f"Claude route:    {config.anthropic_provider}  ->  {'configured' if HAVE_CLAUDE else 'NOT configured: 💲 cells will return errors'}")

## 1. The deterministic checks (free, offline)

The skills are built on plain-Python checks in `ds_workbench.core`. No API key, no network.
They give you the numbers; the skills in parts 2–4 add a model's judgement on top.

The datasets come from `databricks_client` — in local dev mode it serves synthetic tables with
**planted problems**, so you can see whether each check catches them.

### 1a. Leakage scan — which columns separate the target *too* well?
`subscription_churn` hides three leaks behind neutral names: a status code written by the
cancellation flow, an engagement index recomputed after churn, and a refund credit.

In [ ]:
import databricks_client as dc
from ds_workbench.core import leakage_profile

churn = dc.load_sample_dataset("subscription_churn")
display(churn.head())

profile = leakage_profile(churn, target_col="churned")["features"]
scan = pd.DataFrame([
    {"column": col,
     "single_feature_auc": f.get("single_feature_auc"),
     "single_class_values": f.get("single_class_values", "")}
    for col, f in profile.items()
]).sort_values("single_feature_auc", ascending=False)
scan

`engagement_index_v2` has an AUC near 1.0 even though its correlation with the target is low
(the separation is non-linear), and `status_code` has a value that only churned customers carry.
A plain correlation screen misses both. Whether something is truly a leak depends on *when* it's
recorded relative to scoring — that's what the leakage skill reasons about in part 2.

### 1b. Join check — fan-out and data loss, before you run the join

In [ ]:
from ds_workbench.core import join_stats

left, right, key = dc.load_join_scenario("orders_customers_fanout")
s = join_stats(left, right, [key])
print(s["verdict"], "|", "; ".join(s["verdict_reasons"]))
print(f"{len(left):,} orders  ->  {s['result_rows']:,} rows after the join")

### 1c. A/B test — check the split before you trust the effect
Two versions of the same checkout test (true effect: **+1.50** revenue per user). In the second,
a logging bug dropped some of the least engaged treatment users.

In [ ]:
from ds_workbench.core import causal_stats as cs

for name in ["ab_test_clean", "ab_test_srm"]:
    ab = dc.load_sample_dataset(name)
    t = (ab["variant"] == "B").astype(int).to_numpy()
    split = cs.srm(t.sum(), len(t) - t.sum())
    naive = cs.diff_in_means(ab["revenue_30d"].to_numpy(), t)
    cuped = cs.cuped(ab["revenue_30d"].to_numpy(), t, ab["revenue_prev_30d"].to_numpy())
    print(f"{name:15s}  treated share {t.mean():.3f}  SRM p={split['p_value']:.2g}"
          f"  | naive {naive['effect']:+.2f} [{naive['ci_low']:.2f}, {naive['ci_high']:.2f}]"
          f"  | CUPED {cuped['effect']:+.2f} [{cuped['ci_low']:.2f}, {cuped['ci_high']:.2f}]")

On the clean test, CUPED (adjusting for pre-period revenue) shrinks the interval a lot and lands
near +1.50. On the broken one the sample-ratio check fails — so the effect can't be trusted,
however tight its interval looks. The causal skill turns this into a verdict in part 4.

## 2. A predictive workflow: "Who will churn next month?" 💲

A `WorkflowSession` ties the steps together: later skills pick up earlier findings (feature
engineering drops the leaks, model selection factors in data quality) and the session is saved,
so it shows on the app's **Sessions** page next to Workbench runs.

Give the session a **prediction point** — when the model will score. Leakage depends on it.

In [ ]:
from ds_workbench import (WorkflowSession, data_quality, eda, leakage_detector,
                          feature_engineering, model_selector, preflight)

session = WorkflowSession(
    context="Predict which subscribers churn in the next 30 days",
    datasets=["subscription_churn"],
    prediction_point="1st of each month, predicting churn in the next 30 days",
)
df = churn.copy()

### 2a. Data quality first — is the table fit to model on? 💲

In [ ]:
dq = data_quality.assess(df, table_name="subscription_churn", session=session)
dq

### 2b. EDA — what's in here, and what could be the target? 💲

In [ ]:
ed = eda.analyze(df, table_name="subscription_churn", context=session.context, session=session)
ed

### 2c. Leakage — before any training 💲
Each finding says whether it rests on **timing** (the column exists only after the prediction
point) or only on its **statistical** signal. `drop_columns` lists what to remove.

In [ ]:
ld = leakage_detector.check(df, target_col="churned", session=session)
ld

In [ ]:
preflight.check(session=session)        # go / caution / blocked — computed in code, no API call

### 2d. Drop the leaks, then plan features and pick a model 💲
With the session, feature engineering already knows the leakage findings; without one you'd pass
`leakage_findings=ld`. The output includes a runnable PySpark MLlib pipeline.

In [ ]:
X = df.drop(columns=ld.get("drop_columns", []))
print("dropped:", ld.get("drop_columns", []))

fe = feature_engineering.suggest(X, target_col="churned", session=session)
fe

In [ ]:
ms = model_selector.select(X, target_col="churned", problem_context=session.context, session=session)
ms

In [ ]:
session.digest()                        # one-screen summary: verdict, gates, blockers, cost

### 2e. The same column can leak or not — it depends on when you score 💲
In `renewal_timing`, `support_tickets_mtd` counts tickets from the 1st to the 14th. Scoring on the
**1st**, those tickets don't exist yet: leak. Scoring on the **15th**, they do: a legitimate,
strong predictor. Tell the skill when columns are recorded with `column_notes`.

In [ ]:
renewal = dc.load_sample_dataset("renewal_timing")
notes = {
    "support_tickets_mtd": "support tickets opened from the 1st to the 14th of the month",
    "retention_offer_code": "set by the cancellation flow when a subscriber asks to cancel",
    "last_login_days": "days since last login, as of the 1st of the month",
    "churned": "cancelled between the 15th and the end of the month",
}
results = {}
for when in ["1st", "15th"]:
    s = WorkflowSession(context="Renewal churn", datasets=["renewal_timing"], column_notes=notes,
                        prediction_point=f"{when} of each month, predicting whether the subscriber cancels by month end")
    results[when] = (s, leakage_detector.check(renewal, target_col="churned", session=s))
    print(f"scored on the {when}: drop {results[when][1].get('drop_columns')}")

If the statistics flag a column you know is available at scoring time, **you** confirm it — the
model never does. The confirmation is recorded with your name, time and note, and applies only to
this column, dataset and prediction point.

In [ ]:
s15, ld15 = results["15th"]
s15.confirm_available("support_tickets_mtd", note="Counted 1st-14th; we score on the 15th (ETL job 42)")
preflight.check(session=s15)

## 3. A causal investigation: "Did the spring campaign work?" 💲

This is the notebook-first workflow: start from the business question, not a table.

```
frame the question  ->  scout the catalog  ->  you pick tables  ->  verify them  ->  plan
     ->  you build the analysis table  ->  data quality  ->  causal analysis  ->  decide
```
Code enforces the rules (the plan, verification, readiness, the decision status); the model
drafts, searches and explains. The two judgement calls — which tables, and how to build the
analysis table — stay with you.

The catalog here is a synthetic `retail_marketing` fixture: 17 tables including look-alikes
(a forecast, a deprecated table, a temp copy) the scout should reject.

### 3a. Frame the question 💲

In [ ]:
from ds_workbench import problem_framer, data_scout, data_verification

inv = WorkflowSession()
spec = problem_framer.frame(
    "Did our spring campaign increase sales, and should we expand it?",
    context="Marketing chose which customers received it", session=inv)
spec

Read the spec: the **decision** (expand?) is separated from the **analytical question** (did it
work?), the question is typed (causal), and gaps are listed. Confirming it is recommended — it's
recorded with your name.

In [ ]:
if inv.spec:                    # only once framing has produced a spec
    inv.confirm_spec(note="Exposure = received the SPRING26 send")

### 3b. Find the data — metadata only, never rows 💲

In [ ]:
report = data_scout.scout(session=inv)
report

### 3c. You choose the tables, then verify them against the data (free)
Verification runs deterministic checks as aggregates — row counts, freshness, deprecated tags,
how full the mapped columns are, join match rates both ways, and whether the outcome is observed
*after* the exposure. Only this step marks a join verified.

In [ ]:
inv.select_tables(["demo.marketing.campaign_exposure", "demo.crm.customers", "demo.sales.customer_orders"],
                  note="exposure, unit, outcome")
check = data_verification.verify(session=inv)            # add dq=True to also run data quality per table (💲)
check

### 3d. The plan — steps for this question type, the method, and a recipe for your table (free)

In [ ]:
inv.plan()

### 3e. Build the analysis table from the recipe (your step)
On Databricks this would be `spark.table(...)`. Locally, `retail_marketing_data.generate()` returns
the synthetic rows behind each catalog table. One row per customer:
- `treated` = received the SPRING26 send (sent 2 → 29 March 2026; the winter send is a different treatment)
- `outcome` = spend in the 90 days after the campaign; `outcome_pre` = spend in the 90 days before
  (the best proxy for how marketing chose recipients)
- covariates measured before exposure (`loyalty_member`)

The recipe's **Watch** lines matter: some recipients aren't registered customers, and anonymous
till purchases have no customer id — both drop out.

In [ ]:
from ds_workbench.catalogs.retail_marketing_data import generate as load_table   # stand-in for spark.table

customers = load_table("demo.crm.customers")
exposure  = load_table("demo.marketing.campaign_exposure")
orders    = load_table("demo.sales.customer_orders")

spring = exposure[exposure["campaign_id"] == "SPRING26"][["email_hash"]].drop_duplicates()
treated_ids = customers.merge(spring, on="email_hash")["customer_id"]

START, END, WINDOW = pd.Timestamp("2026-03-02"), pd.Timestamp("2026-03-30"), pd.Timedelta(days=90)

def spend(start, end):
    o = orders[(orders["order_date"] >= start) & (orders["order_date"] < end)]
    return o.groupby("customer_id")["net_amount_eur"].sum()

analysis = customers[["customer_id", "loyalty_member"]].copy()
analysis["loyalty_member"] = analysis["loyalty_member"].astype(int)
analysis["treated"] = analysis["customer_id"].isin(treated_ids).astype(int)
analysis["outcome"] = analysis["customer_id"].map(spend(END, END + WINDOW)).fillna(0.0)
analysis["outcome_pre"] = analysis["customer_id"].map(spend(START - WINDOW, START)).fillna(0.0)

print(f"{len(analysis):,} customers, {analysis['treated'].mean():.0%} treated")
print(f"{len(spring) - len(treated_ids):,} SPRING26 recipients aren't registered customers (dropped)")
analysis.groupby("treated")[["outcome", "outcome_pre", "loyalty_member"]].mean()

Treated customers were already spending far more *before* the campaign — the targeting confounds
any naive comparison. Register the table; it's checked against the plan's column contract.

In [ ]:
inv.run_next(analysis)          # registers the analysis table (checked against the contract)

### 3f. Data quality, then the causal analysis — arguments come from the plan 💲

In [ ]:
inv.run_next()                  # data quality on the analysis table

In [ ]:
ci_result = inv.run_next()      # causal inference: observational comparison adjusting for covariates
ci_result

### 3g. Decide — honestly 💲
The **status** is computed in code (not answerable / insufficient evidence / supported with
caveats / supported), and so is ROI across the estimate's confidence interval — only if the effect
is established. The model writes the summary, and every claim must cite the evidence.
Campaign spend was €85k (from `demo.marketing.campaign_costs`).

In [ ]:
inv.decide(inputs={"cost": 85_000,                    # all optional
                   "value_per_outcome_unit": 0.35,    # margin rate: outcome is revenue in EUR
                   "roi_threshold": 0.2})

"Insufficient evidence" is a valid, useful answer — it tells the business what it would take to
know (e.g. a holdout group in the next send) instead of a confident number built on confounded
targeting.

## 4. The rest of the toolbox 💲
Each skill also works standalone, without a session.

### 4a. Join auditor — the verdict from 1b, plus what to do about it

In [ ]:
from ds_workbench import join_auditor
join_auditor.audit(left, right, join_keys=[key], left_name="orders", right_name="customers")

### 4b. Causal inference on A/B traps
- `ab_test_srm` — the broken split from 1c
- `ab_test_bad_control` — adjusting for a metric the treatment itself moved (a mediator) halves the effect

In [ ]:
from ds_workbench import causal_inference
ab_srm = dc.load_sample_dataset("ab_test_srm")
causal_inference.analyze(ab_srm, treatment_col="variant", outcome_col="revenue_30d", treated_value="B",
                         randomized=True, pre_outcome_col="revenue_prev_30d")

In [ ]:
ab_bad = dc.load_sample_dataset("ab_test_bad_control")
causal_inference.analyze(ab_bad, treatment_col="new_onboarding", outcome_col="orders_30d", randomized=True,
                         covariates=["country", "device", "tenure_days", "sessions_first_week"])

### 4c. Difference-in-differences — and the pre-trend trap
A shelf promotion in 20 of 40 stores from 25 May 2026. Here the promo stores were already growing
faster before launch, and the promotion did nothing. A naive DiD reports an effect; the skill
should catch the pre-trend.

In [ ]:
panel = dc.load_sample_dataset("did_pretrend")
causal_inference.did(panel, outcome_col="units_sold", group_col="promo_group", time_col="week_start",
                     treatment_start="2026-05-25", unit_col="store_id", treated_value="promo")

### 4d. NL → PySpark

In [ ]:
from ds_workbench import nl_to_spark
nl_to_spark.generate("What are the top 5 products by total revenue, broken down by region?",
                     {"sales": dc.load_sample_dataset("sales_clean")})

### 4e. MLOps review — is this notebook ready to schedule?

In [ ]:
from ds_workbench import mlops_reviewer
code_to_review = """
df = spark.read.parquet("/mnt/data/transactions/2024-01-15/")
df = df.filter(df.date >= "2023-12-15")
df = df.withColumn("revenue", df.amount * 1.082)
result = df.toPandas()
for i, row in result.iterrows():
    result.at[i, "category"] = "high" if row["revenue"] > 500 else "low"
result.to_csv("/tmp/output.csv")
"""
mlops_reviewer.review(code_to_review)

### 4f. Experiment tracker — paste your metrics after training

In [ ]:
from ds_workbench import experiment_tracker
experiment_tracker.interpret({"train_auc": 0.98, "val_auc": 0.78, "test_auc": 0.77,
                              "model": "gbt", "max_depth": 12, "n_rows": 4000},
                             context="Churn prediction")

## 5. Moving to real data

The skill calls don't change — only where the DataFrame comes from:

| Locally (this notebook) | On Databricks |
|---|---|
| `dc.load_sample_dataset("subscription_churn")` | `spark.table("main.crm.customers").limit(200_000).toPandas()` |
| `load_table("demo.crm.customers")` | `spark.table("<catalog>.crm.customers")` |
| `CATALOG_BACKEND=synthetic` | `CATALOG_BACKEND=unity_catalog` + `CATALOG_ALLOWLIST` + a read-only service principal |

What reaches the model at each step is listed in the README ("What reaches the model"): framing
sends your question, the scout sends metadata only, verification sends nothing, analysis skills
send profiles and statistics of the table you built. Keep real company data off your laptop —
run these skills on real tables inside Databricks.

Every session you ran here is on the app's **Sessions** page (`.venv/bin/python app.py`), and
every Claude call is in the usage ledger on the **Costs** page.